# PSG Transfer Intelligence Evolution
**Pitch Intelligence — Video 1**

From Galacticos to Smart Recruitment: how PSG's Transfer Intelligence Score evolved across eras.

In [ ]:
import sys
sys.path.insert(0, '../src')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib import rcParams

# Pitch Intelligence visual identity
PI_BLACK   = '#0D1B2A'
PI_LIME    = '#C5F135'
PI_WHITE   = '#F0F0F0'
PI_GREY    = '#4A5568'
PI_RED     = '#FF4B4B'
PI_BLUE    = '#4ECDC4'

rcParams['figure.facecolor'] = PI_BLACK
rcParams['axes.facecolor']   = PI_BLACK
rcParams['text.color']       = PI_WHITE
rcParams['axes.labelcolor']  = PI_WHITE
rcParams['xtick.color']      = PI_WHITE
rcParams['ytick.color']      = PI_WHITE
rcParams['axes.edgecolor']   = PI_GREY
rcParams['font.family']      = 'DejaVu Sans'

print('Setup complete — Pitch Intelligence theme loaded.')

## Step 1 — Pull the Data

First run only. After that, use `load_cached()` to avoid hitting FBref repeatedly.

In [ ]:
from scraping.fbref_scraper import pull_all_stats, load_cached
from scraping.transfermarkt_scraper import get_psg_transfers, load_psg_transfers

# ⚠️  Only run these once — they hit external APIs and take ~5 minutes
# fbref_df = pull_all_stats(leagues=['FRA-Ligue 1'], seasons=['2017-2018','2018-2019','2019-2020','2020-2021','2021-2022','2022-2023','2023-2024','2024-2025'])
# transfers_df = get_psg_transfers()

# After first run — use cached:
fbref_df    = load_cached('all_stats_merged.parquet')
transfers_df = load_psg_transfers()

print(f'FBref rows: {len(fbref_df):,}')
print(f'PSG transfers: {len(transfers_df)}')
fbref_df.head(3)

## Step 2 — Run the Full Analysis

In [ ]:
from analysis.psg_transfer_evolution import run_full_analysis

results = run_full_analysis(fbref_df, transfers_df)

era_df    = results['era_averages']
player_df = results['player_scores']
best      = results['best_signings']
worst     = results['worst_signings']

## Chart 1 — The Era Evolution Timeline

Core video chart: TIS by era, showing the clear upward trend.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

eras  = era_df['era'].tolist()
tis   = era_df['avg_tis'].tolist()
spend = era_df['total_spend_m'].tolist()
x     = range(len(eras))

# TIS line
ax.plot(x, tis, color=PI_LIME, linewidth=2.5, marker='o',
        markersize=8, markerfacecolor=PI_LIME, zorder=5)

# shade under the curve
ax.fill_between(x, tis, alpha=0.12, color=PI_LIME)

# spend bars (secondary story)
ax2 = ax.twinx()
ax2.bar(x, spend, color=PI_RED, alpha=0.25, width=0.5, zorder=2)
ax2.set_ylabel('Total Spend (€M)', color=PI_RED, fontsize=10)
ax2.tick_params(colors=PI_RED)
ax2.set_facecolor(PI_BLACK)

# labels
for i, (t, s) in enumerate(zip(tis, spend)):
    ax.annotate(f'{t:.2f}', (i, t), textcoords='offset points',
                xytext=(0, 12), ha='center', color=PI_LIME, fontsize=11, fontweight='bold')

ax.set_xticks(x)
ax.set_xticklabels(eras, fontsize=11)
ax.set_ylabel('Avg Transfer Intelligence Score', color=PI_WHITE, fontsize=11)
ax.set_title('PSG Transfer Intelligence Score by Era', color=PI_WHITE,
             fontsize=16, fontweight='bold', pad=20)
ax.set_facecolor(PI_BLACK)
fig.patch.set_facecolor(PI_BLACK)
ax.spines['top'].set_visible(False)

plt.tight_layout()
plt.savefig('../visualizations/chart1_era_evolution.png', dpi=180, bbox_inches='tight')
plt.show()

## Chart 2 — Spend vs Intelligence Scatter

Each dot = one PSG signing. X = fee paid, Y = TIS. Color = era.
The story: big fees clustered bottom-right, smart signings top-left.

In [ ]:
ERA_COLORS = {
    'Galacticos':       PI_RED,
    'Superstar Chaos':  '#FF8C00',
    'Transition':       PI_GREY,
    'Rebuild':          PI_BLUE,
    'Smart Era':        PI_LIME,
}

fig, ax = plt.subplots(figsize=(12, 7))

for era, grp in player_df.groupby('era'):
    ax.scatter(
        grp['fee_m'], grp['transfer_intelligence_score'],
        c=ERA_COLORS.get(era, PI_GREY), s=80, alpha=0.8,
        label=era, edgecolors='none', zorder=5
    )

# label notable players
notable = player_df.nlargest(3, 'transfer_intelligence_score').index.tolist() + \
          player_df.nsmallest(3, 'transfer_intelligence_score').index.tolist()
for i in notable:
    row = player_df.loc[i]
    ax.annotate(
        row['player'].split()[-1],
        (row['fee_m'], row['transfer_intelligence_score']),
        textcoords='offset points', xytext=(6, 4),
        color=PI_WHITE, fontsize=9
    )

ax.set_xlabel('Transfer Fee (€M)', fontsize=12)
ax.set_ylabel('Transfer Intelligence Score', fontsize=12)
ax.set_title('Fee Paid vs Intelligence Score — Every PSG Signing', color=PI_WHITE,
             fontsize=15, fontweight='bold', pad=20)
ax.legend(facecolor=PI_BLACK, edgecolor=PI_GREY, labelcolor=PI_WHITE)
ax.set_facecolor(PI_BLACK)
fig.patch.set_facecolor(PI_BLACK)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig('../visualizations/chart2_fee_vs_tis_scatter.png', dpi=180, bbox_inches='tight')
plt.show()

## Chart 3 — Average Age at Signing by Era

Shows PSG's shift from proven stars (old) to young talent (upside).

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

colors = [ERA_COLORS.get(e, PI_GREY) for e in era_df['era']]
bars = ax.bar(era_df['era'], era_df['avg_age_at_signing'],
              color=colors, width=0.6, zorder=3)

for bar, val in zip(bars, era_df['avg_age_at_signing']):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.2,
            f'{val:.1f}', ha='center', va='bottom', color=PI_WHITE, fontsize=11)

ax.axhline(y=23, color=PI_LIME, linewidth=1.5, linestyle='--', alpha=0.6)
ax.text(len(era_df) - 0.5, 23.3, 'Age 23 threshold', color=PI_LIME, fontsize=9)

ax.set_ylabel('Avg Age at Signing', fontsize=11)
ax.set_title('PSG Signing Age Profile by Era — Getting Younger', color=PI_WHITE,
             fontsize=14, fontweight='bold', pad=15)
ax.set_facecolor(PI_BLACK)
fig.patch.set_facecolor(PI_BLACK)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.set_ylim(18, 32)

plt.tight_layout()
plt.savefig('../visualizations/chart3_age_by_era.png', dpi=180, bbox_inches='tight')
plt.show()

## Summary Table — Best & Worst Signings

In [ ]:
print('TOP 5 PSG SIGNINGS BY TRANSFER INTELLIGENCE SCORE')
print(best[['player','season','fee_m','age_at_signing','transfer_intelligence_score']].to_string(index=False))
print()
print('BOTTOM 5 PSG SIGNINGS BY TRANSFER INTELLIGENCE SCORE')
print(worst[['player','season','fee_m','age_at_signing','transfer_intelligence_score']].to_string(index=False))